In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.battery import simulate
from src.config import BATTERY_CAPACITY_KWH, BATTERY_POWER_KW, BATTERY_EFF, BATTERY_SELF_DISCHARGE

In [ ]:
# 1. Charge 2 kW for 3 hours from empty, no leak: should store eta * 2 * 3 kWh
out = simulate(np.full(3, 2.0), energy0=0.0, k=0.0)
print("Euler:", round(out["energy_kwh"].iloc[-1], 4), "| hand calculation:", round(BATTERY_EFF * 6, 4))

# 2. Leave a full battery idle for 30 days: should decay by (1 - k) each hour
idle = simulate(np.zeros(24 * 30), energy0=10.0)
print("Euler:", round(idle["energy_kwh"].iloc[-1], 4),
      "| exact decay:", round(10 * np.exp(-BATTERY_SELF_DISCHARGE * 24 * 30), 4))

In [ ]:
idx = pd.date_range("2026-01-14 01:00", periods=24, freq="h")   # hour-ending
request = pd.Series(0.0, index=idx)
request[(idx.hour >= 10) & (idx.hour <= 15)] = 8.0     # ask for more than the 5 kW limit
request[(idx.hour >= 19) & (idx.hour <= 22)] = -4.0    # discharge through the evening

result = simulate(request, energy0=2.0)
result.assign(requested_kw=request).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hours = idx.hour
axes[0].plot(hours, result["soc"] * 100, color="#2a78d6", lw=2)
axes[0].set_title("Battery state of charge"); axes[0].set_ylabel("% full"); axes[0].set_ylim(-5, 105)

axes[1].step(hours, request.values, where="mid", color="#898781", ls="--", label="Requested")
axes[1].step(hours, result["p_actual_kw"].values, where="mid", color="#eb6834", lw=2, label="Actual")
axes[1].axhline(0, color="#0b0b0b", lw=0.8)
axes[1].set_title("Power (positive = charging)"); axes[1].set_ylabel("kW"); axes[1].legend()
for ax in axes:
    ax.set_xlabel("Hour of day (hour-ending, NEM time)")
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(ROOT / "results" / "fig_battery_demo.png", dpi=150)
plt.show()